# Dekompozycja zmienności rozliczeń szkód w hierarchii organizacyjnej ubezpieczyciela za pomocą PROC NESTED

## Streszczenie

Ubezpieczyciel majątkowy i wypadkowy chce wiedzieć, **gdzie** powstaje niespójność w rozliczeniach szkód: czy wynika ona z różnic między regionami geograficznymi, między oddziałami, między poszczególnymi likwidatorami szkód, czy po prostu z szumu między pojedynczymi szkodami. Ten notatnik buduje syntetyczną, w pełni zagnieżdżoną bazę danych o szkodach komunikacyjnych (Region > Oddział > Likwidator > szkoda) i wykorzystuje **PROC NESTED** do przeprowadzenia analizy wariancji z efektami losowymi, szacując składnik wariancji wnoszony przez każdy poziom hierarchii i podając go jako procent całości. Wynik informuje kierownictwo, na który poziom organizacyjny należy ukierunkować działania poprawiające spójność rozliczeń.

## Źródła danych

Wszystkie dane są generowane w pierwszym kroku DATA &mdash; bez plików zewnętrznych, bez sieci. Układ jest **w pełni zagnieżdżony**: każdy oddział należy do dokładnie jednego regionu, każdy likwidator do dokładnie jednego oddziału, a każda szkoda do dokładnie jednego likwidatora.

| Zbiór danych | Wiersze | Zmienna | Typ | Rola | Opis |
|---------|------|----------|------|------|-------------|
| `claims` | 40 | `Region` | char | CLASS (poziom 1, najbardziej zewnętrzny) | Region geograficzny (5 regionów: PN, PD, WS, ZC, CE) |
| | | `Branch` | char | CLASS (poziom 2, zagnieżdżony w Region) | Kod oddziału (2 oddziały na region) |
| | | `Adjuster` | char | CLASS (poziom 3, zagnieżdżony w Branch) | ID likwidatora szkód (2 likwidatorów na oddział) |
| | | `Settlement` | num | VAR (zmienna zależna) | Odszkodowanie wypłacone z tytułu szkody komunikacyjnej, w USD |
| | | `CycleDays` | num | VAR (zmienna zależna) | Liczba dni od pierwszego zgłoszenia szkody do jej rozliczenia |

Struktura syntetyczna: 5 regionów x 2 oddziały x 2 likwidatorów x 2 szkody = 40 obserwacji. Efekt losowy regionu, efekt losowy oddziału zagnieżdżonego w regionie, efekt losowy likwidatora zagnieżdżonego w oddziale oraz reszta na poziomie pojedynczej szkody są nakładane addytywnie za pomocą `rand('NORMAL', ...)`, dzięki czemu składniki wariancji są odtwarzalne. Efekt regionu jest losowany z największym odchyleniem standardowym (2200), więc to *gdzie* obsługiwana jest szkoda stanowi dominujący czynnik. Ziarno ustalone za pomocą `call streaminit(20260531)`. Kompaktowy, w pełni zbalansowany układ zapewnia realne stopnie swobody na każdym poziomie hierarchii.

# Dekompozycja zmienności rozliczeń szkód za pomocą PROC NESTED

Gdy ubezpieczyciel majątkowy i wypadkowy analizuje, ile płaci za rozliczenie szkód komunikacyjnych, często stwierdza dużą zmienność. Część tej zmienności jest nieunikniona (każdy wypadek jest inny), ale część odzwierciedla czynniki **organizacyjne** &mdash; jeden region rozlicza szkody hojniej niż inny, jeden oddział jest bardziej wspaniałomyślny, pojedynczy likwidator jest wartością odstającą.

Dane mają ściśle **zagnieżdżoną** (hierarchiczną) strukturę:

```
Region  ->  Oddział (zagnieżdżony w Regionie)  ->  Likwidator (zagnieżdżony w Oddziale)  ->  pojedyncze szkody
```

Oddział należy do dokładnie jednego regionu, a likwidator do dokładnie jednego oddziału, więc czynniki są zagnieżdżone, a nie skrzyżowane. `PROC NESTED` przeprowadza analizę wariancji z efektami losowymi dokładnie dla takiego układu i szacuje **składnik wariancji** na każdym poziomie, wyrażony jako procent całości. Ten procentowy rozkład odpowiada na pytanie biznesowe: *na który poziom organizacji powinniśmy się skierować, aby rozliczenia były bardziej spójne?*

## Krok 1 &mdash; Wygenerowanie syntetycznej, w pełni zagnieżdżonej bazy szkód

Symulujemy 5 regionów, każdy z 2 oddziałami, każdy z 2 likwidatorami, każdy obsługujący 2 szkody (40 wierszy łącznie). Budujemy odpowiedź z addytywnych efektów losowych, tak aby każdy poziom rzeczywiście wnosił wariancję:

- efekt **regionu** (największy rozrzut &mdash; regiony różnią się najbardziej),
- efekt **oddziału zagnieżdżonego w regionie**,
- efekt **likwidatora zagnieżdżonego w oddziale**,
- oraz **reszta na poziomie szkody** (szum wewnątrz likwidatora).

`call streaminit` ustala ziarno dla powtarzalności; każdy efekt jest losowany za pomocą `rand('NORMAL', mean, sd)`. Efekt regionu wykorzystuje największe odchylenie standardowe, więc powinien odpowiadać za największy udział w całkowitej wariancji. Generujemy również drugą odpowiedź, `CycleDays`, która dzieli tę samą hierarchię, abyśmy mogli później zademonstrować analizę wielu odpowiedzi jednocześnie.

In [1]:
DANE claims;
   CALL streaminit(20260531);
   DŁUGOŚĆ Region $2 Branch $6 Adjuster $9;

   /* Efekt losowy na poziomie regionu: regiony różnią się najbardziej. */
   POWTÓRZ r = 1 TO 5;
      Region = SCAN('PN PD WS ZC CE', r);
      RegionEffect  = rand('NORMAL', 0, 2200);
      RegionCycle   = rand('NORMAL', 0, 11);

      /* Oddział zagnieżdżony w regionie. */
      POWTÓRZ b = 1 TO 2;
         Branch = catx('-', Region, ZAPISZ(b, z2.));
         BranchEffect = rand('NORMAL', 0, 700);
         BranchCycle  = rand('NORMAL', 0, 4);

         /* Likwidator zagnieżdżony w oddziale. */
         POWTÓRZ a = 1 TO 2;
            Adjuster = catx('-', Branch, ZAPISZ(a, z1.));
            AdjEffect = rand('NORMAL', 0, 450);
            AdjCycle  = rand('NORMAL', 0, 2.5);

            /* Pojedyncze szkody obsługiwane przez tego likwidatora. */
            POWTÓRZ claim = 1 TO 2;
               Settlement = 8500
                          + RegionEffect + BranchEffect + AdjEffect
                          + rand('NORMAL', 0, 1100);
               CycleDays  = 21
                          + RegionCycle + BranchCycle + AdjCycle
                          + rand('NORMAL', 0, 6);
               JEŚLI Settlement < 0 WTEDY Settlement = 0;
               JEŚLI CycleDays  < 1 WTEDY CycleDays  = 1;
               WYJŚCIE;
            KONIEC;
         KONIEC;
      KONIEC;
   KONIEC;

   ZACHOWAJ Region Branch Adjuster Settlement CycleDays;
WYKONAJ;


NOTE: DATA claims


NOTE: Wrote claims (40 rows, 5 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds


## Krok 2 &mdash; Sortowanie według hierarchii zagnieżdżenia

`PROC NESTED` wymaga, aby dane wejściowe były **posortowane według zmiennych CLASS w kolejności, w jakiej zostaną wymienione** &mdash; najpierw czynnik najbardziej zewnętrzny. Sortujemy według `Region Branch Adjuster`, aby procedura mogła poprawnie przejść przez hierarchię.

In [2]:
PROCEDURA sort DANE=claims;
   WEDŁUG Region Branch Adjuster;
WYKONAJ;


NOTE: PROC SORT data=claims

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: Read 40 rows from claims.
NOTE: Wrote claims (40 rows, 5 columns).
NOTE: PROC SORT statement used.


## Krok 3 &mdash; Analiza składników wariancji kwoty rozliczenia

Analiza podstawowa. Wymieniamy zmienne CLASS od najbardziej zewnętrznej do najbardziej wewnętrznej (`Region Branch Adjuster`); najbardziej wewnętrzna replikacja &mdash; pojedyncze szkody &mdash; jest automatycznie traktowana jako składnik błędu. `VAR Settlement` wskazuje jedyną zmienną odpowiedzi.

Instrukcja `LABEL` dostarcza przyjaźniejszą nazwę wyświetlaną dla odpowiedzi w nagłówku wyniku. Wynik zawiera współczynniki oczekiwanych średnich kwadratów, tabelę analizy wariancji (z testem F dla każdego składnika wariancji względem zera) oraz oszacowanie **składnika wariancji** na każdym poziomie wraz z jego **procentem całości**.

In [3]:
TYTUŁ 'Składniki wariancji rozliczeń szkód komunikacyjnych';
title2 'Region / Oddział / Likwidator -- ANOVA efektów losowych';

PROCEDURA nested DANE=claims;
   KLASA Region Branch Adjuster;
   ZMIENNA Settlement;
   ETYKIETA Region='Region' Branch='Oddział' Adjuster='Likwidator szkód'
         Settlement = 'Wypłacone odszkodowanie (USD)';
WYKONAJ;

                                  Składniki wariancji rozliczeń szkód komunikacyjnych                                   
                                Region / Oddział / Likwidator -- ANOVA efektów losowych                                 


                       Nested Random Effects Analysis of Variance

Nested Random Effects Analysis of Variance for Variable Wypłacone odszkodowanie (USD)
Variance Source       DF  Sum of Squares       F Value      Pr > F  Error Term   Mean Square  Variance Component    Percent of Total    EMS Coef
Region                 4  62114122.126866          6.71      0.0304   Oddział  15528530.531717  1651824.844989             54.4057      8.0000
Oddział                5  11569658.859028          1.89      0.1829  Likwidator szkód  2313931.771806   272682.828942              8.9813      4.0000
Likwidator szkód      10  12232004.560376          1.22      0.3348     Error  1223200.456038   111582.782346              3.6752      2.0000
Error                 2


NOTE: Option TITLE changed to Składniki wariancji rozliczeń szkód komunikacyjnych.
NOTE: Option TITLE2 changed to Region / Oddział / Likwidator -- ANOVA efektów losowych.
NOTE: PROC NESTED nested ANOVA / variance components

NOTE: PROC NESTED statement used.


## Krok 4 &mdash; Analiza dwóch odpowiedzi jednocześnie

Kierownictwo interesuje się również **czasem cyklu** &mdash; jak długo trwa rozliczenie szkody. Dodajemy `CycleDays` do listy `VAR`. Przy więcej niż jednej zmiennej zależnej `PROC NESTED` dodatkowo raportuje **analizę kowariancji**, pokazującą, jak obie odpowiedzi współzmieniają się na każdym poziomie hierarchii (na przykład, czy regiony, które płacą więcej, rozliczają szkody także wolniej).

In [4]:
TYTUŁ 'Składniki wariancji rozliczenia i czasu likwidacji szkody';
title2 'Dwie zmienne odpowiedzi w tej samej hierarchii zagnieżdżonej';

PROCEDURA nested DANE=claims;
   KLASA Region Branch Adjuster;
   ZMIENNA Settlement CycleDays;
   ETYKIETA Region='Region' Branch='Oddział' Adjuster='Likwidator szkód'
         Settlement = 'Wypłacone odszkodowanie (USD)'
         CycleDays  = 'Czas do rozliczenia (dni)';
WYKONAJ;

                               Składniki wariancji rozliczenia i czasu likwidacji szkody                                
                              Dwie zmienne odpowiedzi w tej samej hierarchii zagnieżdżonej                              


                       Nested Random Effects Analysis of Variance

Nested Random Effects Analysis of Variance for Variable Wypłacone odszkodowanie (USD)
Variance Source       DF  Sum of Squares       F Value      Pr > F  Error Term   Mean Square  Variance Component    Percent of Total    EMS Coef
Region                 4  62114122.126866          6.71      0.0304   Oddział  15528530.531717  1651824.844989             54.4057      8.0000
Oddział                5  11569658.859028          1.89      0.1829  Likwidator szkód  2313931.771806   272682.828942              8.9813      4.0000
Likwidator szkód      10  12232004.560376          1.22      0.3348     Error  1223200.456038   111582.782346              3.6752      2.0000
Error                 2


NOTE: Option TITLE changed to Składniki wariancji rozliczenia i czasu likwidacji szkody.
NOTE: Option TITLE2 changed to Dwie zmienne odpowiedzi w tej samej hierarchii zagnieżdżonej.
NOTE: PROC NESTED nested ANOVA / variance components

NOTE: PROC NESTED statement used.


## Krok 5 &mdash; Widok samej wariancji z opcją AOV

Dla szybkiego podsumowania składników wariancji dla obu odpowiedzi opcja `AOV` ogranicza wynik do statystyk analizy wariancji i **pomija sekcję analizy kowariancji**. To kompaktowy widok, który analityk portfela przeglądałby, gdy potrzebuje jedynie rozkładu wariancji na poszczególne poziomy dla każdej odpowiedzi, bez współzmienności między odpowiedziami.

In [5]:
TYTUŁ 'Tylko składniki wariancji (AOV)';

PROCEDURA nested DANE=claims aov;
   KLASA Region Branch Adjuster;
   ZMIENNA Settlement CycleDays;
   ETYKIETA Region='Region' Branch='Oddział' Adjuster='Likwidator szkód'
         Settlement = 'Wypłacone odszkodowanie (USD)'
         CycleDays  = 'Czas do rozliczenia (dni)';
WYKONAJ;

TYTUŁ;

                                            Tylko składniki wariancji (AOV)                                             
                              Dwie zmienne odpowiedzi w tej samej hierarchii zagnieżdżonej                              


                       Nested Random Effects Analysis of Variance

Nested Random Effects Analysis of Variance for Variable Wypłacone odszkodowanie (USD)
Variance Source       DF  Sum of Squares       F Value      Pr > F  Error Term   Mean Square  Variance Component    Percent of Total    EMS Coef
Region                 4  62114122.126866          6.71      0.0304   Oddział  15528530.531717  1651824.844989             54.4057      8.0000
Oddział                5  11569658.859028          1.89      0.1829  Likwidator szkód  2313931.771806   272682.828942              8.9813      4.0000
Likwidator szkód      10  12232004.560376          1.22      0.3348     Error  1223200.456038   111582.782346              3.6752      2.0000
Error                 2


NOTE: Option TITLE changed to Tylko składniki wariancji (AOV).
NOTE: PROC NESTED nested ANOVA / variance components

NOTE: PROC NESTED statement used.


## Interpretacja wyników

Kolumna **Procent całości** w tabeli analizy wariancji jest najważniejsza. Odczytując ją od góry do dołu, otrzymujemy udział całkowitej zmienności rozliczeń przypisany każdej warstwie organizacyjnej. Dla kwoty rozliczenia analiza daje:

- **Region &mdash; 54,4%** (Pr > F = 0,0304). Dane wygenerowano z największym rozrzutem na poziomie regionu, a analiza to odtwarza: ponad połowa całej zmienności rozliczeń przypada na różnice *między* regionami &mdash; statystycznie istotny dowód, że *gdzie* obsługiwana jest szkoda, stanowi dominujący czynnik.
- **Oddział (w Regionie) &mdash; 9,0%** (Pr > F = 0,18). Skromny dodatkowy udział po zejściu z poziomu regionu do poszczególnych oddziałów; tutaj nieistotny statystycznie.
- **Likwidator (w Oddziale) &mdash; 3,7%** (Pr > F = 0,33). W tej próbie niewiele zmienności można przypisać poszczególnemu likwidatorowi.
- **Błąd &mdash; 32,9%.** Resztowy szum między pojedynczymi szkodami w ramach tego samego likwidatora; jest to nieredukowalna zmienność, której żadna dźwignia organizacyjna nie usunie.

Każdy poziom niesie także **test F (Pr > F)** hipotezy zerowej, że jego składnik wariancji wynosi zero. Niska wartość p dla regionu (0,0304) jest statystycznym dowodem na rzeczywiste systematyczne różnice między regionami, a nie na przypadek losowy.

Odpowiedź czasu cyklu opowiada równoległą historię: **Region odpowiada za 45,8%** zmienności liczby dni do rozliczenia (Pr > F = 0,0448, ponownie istotne), przy czym Oddział i Likwidator wnoszą udziały jednocyfrowe, a reszta niesie 40,1%. Zatem zarówno *ile* płaci ubezpieczyciel, jak i *jak długo* to trwa, są przede wszystkim zdeterminowane przez region.

Analiza dwóch odpowiedzi dodaje **analizę kowariancji**. Tutaj poziom regionu napędza iloczyny krzyżowe, a ogólny **współczynnik korelacji wynosi -0,36** &mdash; zależność ujemna: regiony, które wypłacają wyższe odszkodowania, mają tendencję do *szybszego* ich zamykania, a nie wolniejszego. To użyteczne, nieoczywiste odkrycie &mdash; drogie regiony nie są tymi wolniejszymi.

**Wniosek biznesowy:** ponieważ Region dominuje w rozkładzie procentu całości dla obu odpowiedzi, ubezpieczyciel powinien najpierw ujednolicić wytyczne dotyczące rozliczeń i limity uprawnień *między regionami* &mdash; to tam znajduje się największa, statystycznie istotna niespójność &mdash; zanim zainwestuje w interwencje na poziomie oddziału czy likwidatora. Ujemna korelacja między kwotą rozliczenia a czasem cyklu oznacza, że nie ma jednego regionu, który byłby jednocześnie najdroższy i najwolniejszy, więc oba problemy można rozwiązywać osobnymi, regionalnie ukierunkowanymi programami. PROC NESTED zamienia niejasne wrażenie "nasze rozliczenia są niespójne" w praktyczne, warstwa po warstwie, wskazanie źródła tej niespójności.